# Legacy notebook (archived)

This notebook is kept only for reference. Use `00_start_here.ipynb`, `01_eda.ipynb`, and `02_modeling.ipynb` instead.


In [ ]:
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / 'data' / 'Automobile_insurance_fraud_cleaned.csv'
df = pd.read_csv(DATA_PATH)
sns.countplot(x='incident_severity', hue='fraud_reported', data = df)
plt.title("Signal: Major damage following high fraud")


In [ ]:
plt.figure(figsize=(12,6))
sns.countplot(x='insured_hobbies', hue='fraud_reported', data=df)
plt.xticks(rotation=45)


In [ ]:
sns.boxplot(x='fraud_reported', y='total_claim_amount',data=df)


In [ ]:
age_bins = [0, 25, 35, 45, 65]
age_labels = ['18-25', '26-35','36-45','46-65']
df['age_group'] = pd.cut(df['age'], bins=age_bins, labels=age_labels, include_lowest=True)

tenure_bins=[0,60,180,300,500]
tenure_labels=['New (<5yr)', 'Mid (5-15yr)', 'Long (15-25yr)','Legacy (>25yr)']
df['tenure_group'] = pd.cut(df['months_as_customer'], bins=tenure_bins, labels=tenure_labels, include_lowest=True)

plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
sns.barplot(x='age_group', y='fraud_reported', data=df, errorbar=None, palette='Reds')
plt.title("Fraud rates by age group")
plt.xticks(rotation=45)

plt.subplot(1,2,2)
sns.barplot(x='tenure_group', y='fraud_reported', data=df, errorbar=None, palette='Blues')
plt.title('Fraud rates by tenure of customer')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
import sys
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
 sys.path.append(str(SRC_PATH))
from preprocessing import encode_categorical_features, scale_numerical_features
df_encoded = encode_categorical_features(df)
print(df_encoded.columns.tolist())


In [ ]:
numerical_cols = [
        'months_as_customer', 'age', 'policy_annual_premium',
        'umbrella_limit', 'capital-gains', 'capital-loss',
        'incident_hour_of_the_day', 'number_of_vehicles_involved',
        'bodily_injuries', 'witnesses', 'total_claim_amount',
        'injury_claim', 'property_claim', 'vehicle_claim', 'days_to_incident'
    ]
cols_to_scale = [col for col in numerical_cols if col in df_encoded.columns]

df_scaled = scale_numerical_features(df_encoded)
print(df_scaled[cols_to_scale].describe().round(2))


In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from collections import Counter

feature_drop_cols = ['fraud_reported', 'policy_bind_date', 'incident_date']
x = df_scaled.drop(columns=feature_drop_cols, errors='ignore')
y = df_scaled['fraud_reported']

x_train, x_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42, stratify=y)

print(f"Label allocation before SMOTE: {Counter(y_train)}")

smote = SMOTE(random_state=42)
x_train_res, y_train_res = smote.fit_resample(x_train, y_train)

print(f"Label allocation after SMOTE: {Counter(y_train_res)}")


In [ ]:
import sys
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
 sys.path.append(str(SRC_PATH))
from modeling import train_baseline_rf, train_xgboost
from sklearn.metrics import confusion_matrix, classification_report, recall_score
rf_model, rf_pred = train_baseline_rf(x_train, y_train, x_test, y_test)

# 2. Run XGBoost (legacy)
xgb_model, xgb_pred = train_xgboost(x_train, y_train, x_test, y_test)
cm_rf = confusion_matrix(y_test, rf_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix of Random Forest - Baseline Model')
plt.show()

# Print detailed report
print("\nDetailed Classification Report:")
print(classification_report(y_test, rf_pred))

cm_xgb = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix of XGBoost - Advanced Model')
plt.show()

# Print detailed report
print("\nDetailed Classification Report of Random Forest:")
print(classification_report(y_test, rf_pred))

print("\nDetailed Classification Report of XGBoost:")
print(classification_report(y_test, xgb_pred))

print("\n--- COMPARISON ---")
print(f"Random Forest Recall: {recall_score(y_test, rf_pred):.4f}")
print(f"XGBoost Recall:       {recall_score(y_test, xgb_pred):.4f}")


In [ ]:
import sys
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
 sys.path.append(str(SRC_PATH))
from modeling import tune_xgboost, plot_pr_comparision, save_model

best_xgb_model = tune_xgboost(x_train, y_train)
y_pred = best_xgb_model.predict(x_test)

models_to_compare = {
    'XGBoost Baseline': xgb_model,
    'XGBoost Tuned (GridSearch)': best_xgb_model
}   
plot_pr_comparision(models_to_compare, x_test, y_test)
save_model(best_xgb_model, 'best_xgboost_v1.pkl')


In [ ]:
import sys
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
 sys.path.append(str(SRC_PATH))
from modeling import evaluate_model_performance, plot_all_curves
metrics = evaluate_model_performance(best_xgb_model, x_test, y_test)
plot_all_curves(best_xgb_model, x_test, y_test)
